# K-Nearest Neighbour Classifier

7-class classifier. NOTE: KNN stores the entire training set to make predictions

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn joblib openpyxl matplotlib skl2onnx onnx


In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import MiniBatchKMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler



In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

BASE_PATH = (
    r"../../data"
    r"/Motor_data_model"
)

SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s",
    "PWM",
    "Irms",
    "Vrms",
    "Current_Variance",
    "Current_Crest_Factor",
    "Vib_RMS",
    "Vib_Min",
    "Vib_Q1",
    "Vib_Median",
    "Vib_Q3",
    "Vib_Max",
    "Vib_Variance",
    "Vib_Skewness",
]

FINAL_LABELS = [
    "motor_off",
    "turning_off",
    "turning_on",
    "normal",
    "electrical_fault",
    "mechanical_fault",
    "thermal_fault",
]

LABEL_TO_ID = {
    label: index
    for index, label in enumerate(FINAL_LABELS)
}

N_FEATURES = len(FEATURE_COLUMNS)
N_CLASSES = len(FINAL_LABELS)

WINDOW_LEN = 30
WINDOW_STEP = 10
INPUT_SIZE = WINDOW_LEN * N_FEATURES
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
# Number of representative KNN samples stored per class
PROTOTYPES_PER_CLASS = 30

OUTPUT_DIR = "knn_output"
CV_DIR = os.path.join(
    OUTPUT_DIR,
    "cross_validation",
)
FINAL_DIR = os.path.join(
    OUTPUT_DIR,
    "final_model",
)
MANUAL_TEST_DIR = os.path.join(
    OUTPUT_DIR,
    "manual_test_results",
)

for folder in [
    CV_DIR,
    FINAL_DIR,
    MANUAL_TEST_DIR,
]:
    os.makedirs(
        folder,
        exist_ok=True,
    )

print("Model: K-Nearest Neighbors Classifier")
print("Input:", WINDOW_LEN, "x", N_FEATURES)
print("Flattened input size:", INPUT_SIZE)
print("Classes:", FINAL_LABELS)

In [ ]:
# ==========================================================
# CELL 3 — map each file path to one class
# ==========================================================
def normalized_path(path):
    return str(path).replace(
        "\\",
        "/",
    ).lower()


def label_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    if (
        "motor_being_off" in path_text
        or "turning_off" in path_text
        or "transition_off" in path_text
        or "off_spikes" in path_text
    ):
        return "turning_off"

    if (
        "motor_being_on" in path_text
        or "turning_on" in path_text
        or "transition_on" in path_text
        or "on_spikes" in path_text
    ):
        return "turning_on"

    if (
        "/motor_off/" in path_text
        or "motor_off" in stem
    ):
        return "motor_off"

    if (
        "electrical fault" in path_text
        or "electrical_fault" in path_text
        or "abnormal_current" in path_text
    ):
        return "electrical_fault"

    if (
        "mechanical fault" in path_text
        or "mechanical_fault" in path_text
        or "abnormal_vibration" in path_text
    ):
        return "mechanical_fault"

    if (
        "thermal fault" in path_text
        or "thermal_fault" in path_text
        or "abnormal_temp" in path_text
    ):
        return "thermal_fault"

    if (
        "/normal/" in path_text
        or stem.startswith("normal_")
        or stem == "normal"
    ):
        return "normal"

    return None


In [ ]:
# ==========================================================
# CELL 4 — discover and read CSV / Excel files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(
        os.path.join(
            base_path,
            "**",
            "*.xlsx",
        ),
        recursive=True,
    )

    files += glob.glob(
        os.path.join(
            base_path,
            "**",
            "*.csv",
        ),
        recursive=True,
    )

    return sorted(
        set(files)
    )


def read_one(path):
    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    missing_columns = [
        column
        for column in FEATURE_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in {path.name}: "
            f"{missing_columns}"
        )

    clean = df[
        FEATURE_COLUMNS
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    clean = clean / SCALE_DIVISOR

    clean = clean.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    clean = clean.dropna(
    ).reset_index(
        drop=True
    )

    if len(clean) < WINDOW_LEN:
        raise ValueError(
            f"{path.name} has only "
            f"{len(clean)} valid rows."
        )

    return clean



In [ ]:
# ==========================================================
# CELL 5 — load labeled sessions
# ==========================================================
files = discover_files(
    BASE_PATH
)

if not files:
    raise RuntimeError(
        f"No CSV or Excel files found under: "
        f"{BASE_PATH}"
    )

sessions = []

for file_path in files:
    label = label_for(
        file_path
    )

    if label is None:
        print(
            "SKIP unrecognized:",
            os.path.relpath(
                file_path,
                BASE_PATH,
            ),
        )
        continue

    try:
        df = read_one(
            file_path
        )
    except Exception as error:
        print(
            "SKIP read error:",
            os.path.relpath(
                file_path,
                BASE_PATH,
            ),
            "|",
            error,
        )
        continue

    sessions.append({
        "X": df.to_numpy(
            dtype=np.float32
        ),
        "file": os.path.relpath(
            file_path,
            BASE_PATH,
        ),
        "label": label,
        "class_id": LABEL_TO_ID[
            label
        ],
        "rows": len(df),
    })

if not sessions:
    raise RuntimeError(
        "No labeled sessions were loaded."
    )

class_counts = pd.Series([
    session["label"]
    for session in sessions
]).value_counts().reindex(
    FINAL_LABELS,
    fill_value=0,
)

print("\nFILES PER CLASS")
print(class_counts)

missing_classes = [
    label
    for label in FINAL_LABELS
    if class_counts[label] == 0
]

if missing_classes:
    raise RuntimeError(
        f"No files found for classes: "
        f"{missing_classes}"
    )


In [ ]:
# ==========================================================
# CELL 6 — create flattened 30 × 14 windows
# ==========================================================
def make_windows_from_array(
    X_raw,
):
    starts = np.arange(
        0,
        len(X_raw) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_raw[
            start:start + WINDOW_LEN
        ]
        for start in starts
    ]).astype(
        np.float32
    )

    flattened = windows.reshape(
        len(windows),
        INPUT_SIZE,
    )

    return (
        flattened,
        starts,
    )


def build_dataset_from_sessions(
    selected_sessions,
):
    all_windows = []
    all_labels = []
    all_files = []
    all_starts = []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(
            session["X"]
        )

        if windows is None:
            continue

        all_windows.append(
            windows
        )

        all_labels.extend(
            [session["class_id"]]
            * len(windows)
        )

        all_files.extend(
            [session["file"]]
            * len(windows)
        )

        all_starts.extend(
            starts.tolist()
        )

    if not all_windows:
        raise RuntimeError(
            "No windows were created."
        )

    return (
        np.concatenate(
            all_windows,
            axis=0,
        ),
        np.asarray(
            all_labels,
            dtype=np.int64,
        ),
        np.asarray(
            all_files,
            dtype=object,
        ),
        np.asarray(
            all_starts,
            dtype=np.int64,
        ),
    )
    


In [ ]:
# ==========================================================
# CELL 7 — flattened feature names
# ==========================================================
FLAT_FEATURE_NAMES = []

for time_index in range(
    WINDOW_LEN
):
    for feature_name in FEATURE_COLUMNS:
        FLAT_FEATURE_NAMES.append(
            f"t{time_index:02d}_"
            f"{feature_name}"
        )

In [ ]:
# ==========================================================
# CELL 8 — build compact prototype-based KNN classifier
# ==========================================================
def build_knn_classifier():
    return KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        algorithm="brute",
        leaf_size=30,
        p=2,
        metric="minkowski",
        n_jobs=-1,
    )


def create_class_prototypes(
    X_scaled,
    y,
):
    prototype_X = []
    prototype_y = []

    for class_id in range(
        N_CLASSES
    ):
        class_X = X_scaled[
            y == class_id
        ]

        if len(class_X) == 0:
            raise RuntimeError(
                f"No training samples found for class "
                f"{FINAL_LABELS[class_id]}."
            )

        number_of_prototypes = min(
            PROTOTYPES_PER_CLASS,
            len(class_X),
        )

        if len(class_X) <= PROTOTYPES_PER_CLASS:
            class_prototypes = class_X
        else:
            clustering = MiniBatchKMeans(
                n_clusters=number_of_prototypes,
                random_state=SEED,
                batch_size=1024,
                n_init=10,
                max_iter=300,
            )

            clustering.fit(
                class_X
            )

            class_prototypes = (
                clustering.cluster_centers_
                .astype(np.float32)
            )

        prototype_X.append(
            class_prototypes
        )

        prototype_y.append(
            np.full(
                len(class_prototypes),
                class_id,
                dtype=np.int64,
            )
        )

        print(
            FINAL_LABELS[class_id],
            "| original:",
            len(class_X),
            "| stored prototypes:",
            len(class_prototypes),
        )

    return (
        np.concatenate(
            prototype_X,
            axis=0,
        ).astype(np.float32),
        np.concatenate(
            prototype_y,
            axis=0,
        ).astype(np.int64),
    )


def fit_compact_knn(
    X_train,
    y_train,
):
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    ).astype(np.float32)

    X_prototypes, y_prototypes = (
        create_class_prototypes(
            X_train_scaled,
            y_train,
        )
    )

    classifier = build_knn_classifier()

    classifier.fit(
        X_prototypes,
        y_prototypes,
    )

    model = Pipeline([
        (
            "scaler",
            scaler,
        ),
        (
            "classifier",
            classifier,
        ),
    ])

    return (
        model,
        len(X_prototypes),
    )


temporary_classifier = build_knn_classifier()

print(
    "Compact KNN configuration:"
)

print(
    temporary_classifier.get_params()
)

print(
    "Maximum prototypes per class:",
    PROTOTYPES_PER_CLASS,
)

print(
    "Maximum total stored prototypes:",
    PROTOTYPES_PER_CLASS * N_CLASSES,
)

del temporary_classifier

In [ ]:
# ==========================================================
# CELL 9 — five-fold file-wise cross-validation
# ==========================================================
cv_table = pd.DataFrame({
    "label": [
        session["class_id"]
        for session in sessions
    ],
    "group": [
        session["file"]
        for session in sessions
    ],
})

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

cv_metrics = []
all_true = []
all_pred = []
all_window_predictions = []

for fold, (
    train_index,
    test_index,
) in enumerate(
    splitter.split(
        cv_table,
        cv_table["label"],
        groups=cv_table["group"],
    ),
    start=1,
):
    print("\n" + "=" * 60)
    print("FOLD", fold)
    print("=" * 60)

    train_sessions = [
        sessions[index]
        for index in train_index
    ]

    test_sessions = [
        sessions[index]
        for index in test_index
    ]

    X_train, y_train, _, _ = (
        build_dataset_from_sessions(
            train_sessions
        )
    )

    (
        X_test,
        y_test,
        test_files,
        test_starts,
    ) = build_dataset_from_sessions(
        test_sessions
    )

    model, stored_prototypes = fit_compact_knn(
    X_train,
    y_train,
    )

    print(
        "Stored prototypes in this fold:",
        stored_prototypes,
    )

    predictions = model.predict(
        X_test
    )

    probabilities = model.predict_proba(
        X_test
    )

    confidence = np.max(
        probabilities,
        axis=1,
    )

    all_true.extend(
        y_test.tolist()
    )

    all_pred.extend(
        predictions.tolist()
    )

    fold_metrics = {
        "fold": fold,
        "accuracy": accuracy_score(
            y_test,
            predictions,
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_test,
                predictions,
            ),
        "macro_precision":
            precision_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0,
            ),
        "macro_recall":
            recall_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0,
            ),
        "macro_f1":
            f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0,
            ),
    }

    cv_metrics.append(
        fold_metrics
    )

    fold_predictions = pd.DataFrame({
        "fold": fold,
        "file": test_files,
        "start_row": test_starts,
        "true_class_id": y_test,
        "true_label": [
            FINAL_LABELS[index]
            for index in y_test
        ],
        "pred_class_id": predictions,
        "pred_label": [
            FINAL_LABELS[index]
            for index in predictions
        ],
        "confidence": confidence,
    })

    all_window_predictions.append(
        fold_predictions
    )

    print(
        pd.Series(
            fold_metrics
        )
    )



In [ ]:
# ==========================================================
# CELL 10 — combined cross-validation report
# ==========================================================
cv_metrics_df = pd.DataFrame(
    cv_metrics
)

cv_predictions_df = pd.concat(
    all_window_predictions,
    ignore_index=True,
)

cv_metrics_df.to_csv(
    os.path.join(
        CV_DIR,
        "cv_metrics.csv",
    ),
    index=False,
)

cv_predictions_df.to_csv(
    os.path.join(
        CV_DIR,
        "cv_window_predictions.csv",
    ),
    index=False,
)

all_true = np.asarray(
    all_true,
    dtype=np.int64,
)

all_pred = np.asarray(
    all_pred,
    dtype=np.int64,
)

overall_metrics = {
    "accuracy": accuracy_score(
        all_true,
        all_pred,
    ),
    "balanced_accuracy":
        balanced_accuracy_score(
            all_true,
            all_pred,
        ),
    "macro_precision":
        precision_score(
            all_true,
            all_pred,
            average="macro",
            zero_division=0,
        ),
    "macro_recall":
        recall_score(
            all_true,
            all_pred,
            average="macro",
            zero_division=0,
        ),
    "macro_f1":
        f1_score(
            all_true,
            all_pred,
            average="macro",
            zero_division=0,
        ),
}

print(
    "\nOVERALL CROSS-VALIDATION RESULTS"
)

print(
    pd.Series(
        overall_metrics
    )
)

print(
    classification_report(
        all_true,
        all_pred,
        labels=np.arange(
            N_CLASSES
        ),
        target_names=FINAL_LABELS,
        zero_division=0,
    )
)

pd.DataFrame([
    overall_metrics
]).to_csv(
    os.path.join(
        CV_DIR,
        "overall_metrics.csv",
    ),
    index=False,
)

ConfusionMatrixDisplay(
    confusion_matrix(
        all_true,
        all_pred,
        labels=np.arange(
            N_CLASSES
        ),
    ),
    display_labels=FINAL_LABELS,
).plot(
    values_format="d",
    xticks_rotation=45,
)

plt.title(
    "KNN Classifier Cross-Validation"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        CV_DIR,
        "confusion_matrix.png",
    ),
    dpi=180,
)

plt.show()

In [ ]:
# ==========================================================
# CELL 11 — train compact final KNN on all sessions
# ==========================================================
X_all, y_all, _, _ = (
    build_dataset_from_sessions(
        sessions
    )
)

final_model, final_prototype_count = (
    fit_compact_knn(
        X_all,
        y_all,
    )
)

print(
    "Original training windows:",
    len(X_all),
)

print(
    "Stored KNN prototypes:",
    final_prototype_count,
)

print(
    "Training-window reduction:",
    round(
        100.0
        * (
            1.0
            - final_prototype_count
            / len(X_all)
        ),
        2,
    ),
    "%",
)

In [ ]:
# ==========================================================
# CELL 12 — save model, labels, and metadata
# ==========================================================
MODEL_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "knn_classifier.pkl",
    )
)

LABELS_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "knn_labels.txt",
    )
)

METADATA_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "knn_metadata.csv",
    )
)

joblib.dump(
    final_model,
    MODEL_PATH,
    compress=3,
)

with open(
    LABELS_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write(
        "\n".join(
            FINAL_LABELS
        )
    )

pd.DataFrame({
    "flat_feature_id": np.arange(
        INPUT_SIZE
    ),
    "flat_feature_name":
        FLAT_FEATURE_NAMES,
}).to_csv(
    METADATA_PATH,
    index=False,
)

print("Saved model:", MODEL_PATH)

print(
    "Model size KB:",
    round(
        os.path.getsize(
            MODEL_PATH
        ) / 1024,
        2,
    ),
)

print("Saved labels:", LABELS_PATH)
print("Saved metadata:", METADATA_PATH)


In [ ]:
# ==========================================================
# CELL 13 — verify class order
# ==========================================================
trained_classifier = final_model.named_steps[
    "classifier"
]

expected_classes = np.arange(
    N_CLASSES,
    dtype=np.int64,
)

if not np.array_equal(
    trained_classifier.classes_,
    expected_classes,
):
    raise RuntimeError(
        "KNN class order does not match FINAL_LABELS. "
        f"Model classes: {trained_classifier.classes_}"
    )

print(
    "Verified class order:",
    trained_classifier.classes_,
)


In [ ]:
# ==========================================================
# CELL 14 — nearest-neighbor distance report
# ==========================================================
trained_scaler = final_model.named_steps[
    "scaler"
]

X_all_scaled = trained_scaler.transform(
    X_all
)

number_check = min(
    100,
    len(X_all_scaled),
)

neighbor_distances, neighbor_indices = (
    trained_classifier.kneighbors(
        X_all_scaled[
            :number_check
        ],
        n_neighbors=5,
        return_distance=True,
    )
)

distance_report = pd.DataFrame({
    "sample_id": np.arange(
        number_check
    ),
    "nearest_distance":
        neighbor_distances[:, 0],
    "mean_5_neighbor_distance":
        np.mean(
            neighbor_distances,
            axis=1,
        ),
    "maximum_5_neighbor_distance":
        np.max(
            neighbor_distances,
            axis=1,
        ),
})

distance_report.to_csv(
    os.path.join(
        FINAL_DIR,
        "neighbor_distance_report.csv",
    ),
    index=False,
)

print(
    "\nNEIGHBOR DISTANCE SUMMARY"
)

print(
    distance_report.describe()
)



In [ ]:
# ==========================================================
# CELL 15 — export complete KNN pipeline to ONNX
# ==========================================================
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

ONNX_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "knn_complete_pipeline.onnx",
    )
)

onnx_model = convert_sklearn(
    final_model,
    initial_types=[
        (
            "float_input",
            FloatTensorType([
                None,
                INPUT_SIZE,
            ]),
        )
    ],
    target_opset=13,
)

with open(
    ONNX_PATH,
    "wb",
) as file:
    file.write(
        onnx_model.SerializeToString()
    )

print(
    "Saved complete ONNX:",
    ONNX_PATH,
)

print(
    "ONNX size KB:",
    round(
        os.path.getsize(
            ONNX_PATH
        ) / 1024,
        2,
    ),
)

print(
    "The ONNX contains both StandardScaler and KNN."
)


In [ ]:
# ==========================================================
# CELL 16 — model size and STM32 deployment warning
# ==========================================================
training_storage_bytes = (
    X_all.shape[0]
    * X_all.shape[1]
    * 4
    + y_all.shape[0]
    * 4
)

print(
    "Approximate raw KNN training storage MB:",
    round(
        training_storage_bytes
        / (1024 * 1024),
        2,
    ),
)

print(
    "Important: KNN stores all training windows. "
    "The ONNX file can be much larger and slower than "
    "Logistic Regression, Linear SVM, Decision Tree, "
    "Random Forest, and XGBoost."
)


In [ ]:
# ==========================================================
# CELL 17 — manual classification test
# ==========================================================
def manual_classification_prediction(
    test_file,
):
    model_path = os.path.abspath(
        os.path.join(
            "knn_output",
            "final_model",
            "knn_classifier.pkl",
        )
    )

    print(
        "Loading model from:",
        model_path,
    )

    if not os.path.exists(
        model_path
    ):
        raise FileNotFoundError(
            f"Model file not found:\n"
            f"{model_path}\n\n"
            "Run Cell 11 and Cell 12 first."
        )

    model = joblib.load(
        model_path
    )

    test_df = read_one(
        test_file
    )

    X_test, starts = make_windows_from_array(
        test_df.to_numpy(
            dtype=np.float32
        )
    )

    if (
        X_test is None
        or len(X_test) == 0
    ):
        raise RuntimeError(
            "The selected file does not contain enough "
            "valid rows for one 30-row window."
        )

    predictions = model.predict(
        X_test
    )

    probabilities = model.predict_proba(
        X_test
    )

    confidence = np.max(
        probabilities,
        axis=1,
    )

    result = pd.DataFrame({
        "window": np.arange(
            len(X_test)
        ),
        "start_row": starts,
        "time_sec": (
            starts + WINDOW_LEN / 2.0
        ) * ROW_PERIOD_SEC,
        "pred_class_id":
            predictions,
        "pred_label": [
            FINAL_LABELS[
                int(index)
            ]
            for index in predictions
        ],
        "confidence":
            confidence,
    })

    for class_id, label in enumerate(
        FINAL_LABELS
    ):
        result[
            f"probability_{label}"
        ] = probabilities[
            :,
            class_id,
        ]

    majority_class_id = int(
        result[
            "pred_class_id"
        ].value_counts(
        ).idxmax()
    )

    majority_label = FINAL_LABELS[
        majority_class_id
    ]

    stem = Path(
        test_file
    ).stem

    manual_test_dir = os.path.abspath(
        os.path.join(
            "knn_output",
            "manual_test_results",
        )
    )

    os.makedirs(
        manual_test_dir,
        exist_ok=True,
    )

    output_csv = os.path.join(
        manual_test_dir,
        f"manual_test_{stem}.csv",
    )

    result.to_csv(
        output_csv,
        index=False,
    )

    print("\n" + "=" * 60)
    print("MANUAL KNN TEST")
    print("=" * 60)
    print("File:", test_file)
    print("Total windows:", len(result))
    print(
        "Majority prediction:",
        majority_label,
    )
    print(
        "Mean confidence:",
        float(
            np.mean(
                confidence
            )
        ),
    )
    print("Saved CSV:", output_csv)

    print(
        "\nWindow predictions:"
    )

    print(
        result[
            "pred_label"
        ].value_counts()
    )

    plt.figure(
        figsize=(12, 4)
    )

    plt.step(
        result["time_sec"],
        result["pred_class_id"],
        where="mid",
    )

    plt.yticks(
        np.arange(
            N_CLASSES
        ),
        FINAL_LABELS,
    )

    plt.xlabel("Time [s]")
    plt.ylabel("Predicted class")

    plt.title(
        f"KNN — {stem}"
    )

    plt.tight_layout()

    timeline_path = os.path.join(
        manual_test_dir,
        f"manual_test_{stem}_timeline.png",
    )

    plt.savefig(
        timeline_path,
        dpi=150,
    )

    plt.show()

    print(
        "Saved timeline:",
        timeline_path,
    )

    return result



In [ ]:
# ==========================================================
# CELL 18 — select manual test file
# ==========================================================
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes(
    "-topmost",
    True,
)

TEST_FILE = filedialog.askopenfilename(
    title=(
        "Select motor CSV or Excel file"
    ),
    filetypes=[
        (
            "Excel files",
            "*.xlsx",
        ),
        (
            "CSV files",
            "*.csv",
        ),
        (
            "All files",
            "*.*",
        ),
    ],
)

root.destroy()

if TEST_FILE:
    manual_result = (
        manual_classification_prediction(
            TEST_FILE
        )
    )
else:
    print(
        "No file selected."
    )